# Instrument Calibration

This notebook covers the calibration workflow for both thermistor sensor types.

**Part 1 — Geoprecision thermistor chain calibration**
- Load thermistor chain data from calibration runs
- Compute 0-degree offsets from ice-bath measurements

**Part 2 — Tynitag NTC logger calibration**
- Compute 0-degree offsets from ice-bath runs for all 15 loggers
- Reliability experiments (Exp1, Exp2, Exp3)
- Statistical comparison of Tynitag vs. Geoprecision calibration offsets


---

## Part 1 — Geoprecision Thermistor Chain Calibration

## Import required Libraries and Modules

In [ ]:
from config import ICETEMP_ROOT
import sys
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import cmcrameri.cm as cmc

mpl.rcParams.update({'font.family': 'Arial'})

# Add project root to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.thermistor_processing import *
from src.thermistor_plotting import *
from src.gpr_plotting import *
from calibration.calibration_utils import compute_and_save_offsets
from calibration.thermistor_calibration import *
from calibration import thermistor_chains_icebath_references as chains


In [ ]:
# root dir
root_dir = ICETEMP_ROOT + "/"

# set main calibration data dir
cal_dir = os.path.join(root_dir, "thermistor_chains", "calibration_data") + "/"

# set chain calibration dir
A551FD_dir = cal_dir + "A551FD/raw/"
A551FE_dir = cal_dir + "A551FE/raw/"
A55200_dir = cal_dir + "A55200/raw/"
A55201_dir = cal_dir + "A55201/raw/"
A55202_dir = cal_dir + "A55202/raw/"
A55203_dir = cal_dir + "A55203/raw/"
A55204_dir = cal_dir + "A55204/raw/"
A55205_dir = cal_dir + "A55205/raw/"

# set maximum measurement depths
A551FE_depth = 45.0 # AH1 -> borehole depth is 50.6m
A55204_depth = 20.3 # AH2
A55205_depth = 18.3 # AH3 -> borehole depth is 58.3m
A551FD_depth = 29.0 # HL1
A55203_depth = 21.5 # HL2 
A55200_depth = 21.5 # HL3
A55201_depth = 38.3 # CH1
A55202_depth = 17.0 # CH2

# generate calibration data objects
A551FE_cal_data = ThermistorData(A551FE_dir + "A551FE_20250729123855.csv",",",A551FE_depth)   
A55204_cal_data = ThermistorData(A55204_dir + "A55204_20250729123756.csv",",",A55204_depth)   
A55205_cal_data = ThermistorData(A55205_dir + "A55205_20250729123810.csv",",",A55205_depth)   
A551FD_cal_data = ThermistorData(A551FD_dir + "A551FD_20250729123824.csv",",",A551FD_depth)
A55203_cal_data = ThermistorData(A55203_dir + "A55203_20250729123726.csv",",",A55203_depth)   
A55200_cal_data = ThermistorData(A55200_dir + "A55200_20250729123642.csv",",",A55200_depth)   
A55201_cal_data = ThermistorData(A55201_dir + "A55201_20250729123655.csv",",",A55201_depth)   
A55202_cal_data = ThermistorData(A55202_dir + "A55202_20250729123712.csv",",",A55202_depth)   

## Load thermistor chain data

In [ ]:
# get calibration data for each thermistor chain
A551FD_cal_df = A551FD_cal_data.get_chain_data('28.07.2025 13:30:00','28.07.2025 14:30:00')
A551FE_cal_df = A551FE_cal_data.get_chain_data('28.07.2025 12:00:00','28.07.2025 14:00:00')
A55200_cal_df = A55200_cal_data.get_chain_data('29.07.2025 10:00:00','29.07.2025 13:00:00')
A55201_cal_df = A55201_cal_data.get_chain_data('28.07.2025 09:00:00','28.07.2025 12:00:00')
A55202_cal_df = A55202_cal_data.get_chain_data('28.07.2025 14:00:00','28.07.2025 16:00:00')
A55203_cal_df = A55203_cal_data.get_chain_data('28.07.2025 08:00:00','28.07.2025 10:00:00')
A55204_cal_df = A55204_cal_data.get_chain_data('25.07.2025 12:00:00','25.07.2025 14:00:00')
A55205_cal_df = A55205_cal_data.get_chain_data('25.07.2025 13:00:00','25.07.2025 15:00:00')


In [ ]:
# calculate zero degree offsets
all_offsets = {
    "A551FD": calculate_chain_zero_degree_offsets(A551FD_cal_df),
    "A551FE": calculate_chain_zero_degree_offsets(A551FE_cal_df),
    "A55200": calculate_chain_zero_degree_offsets(A55200_cal_df),
    "A55201": calculate_chain_zero_degree_offsets(A55201_cal_df),
    "A55202": calculate_chain_zero_degree_offsets(A55202_cal_df),
    "A55203": calculate_chain_zero_degree_offsets(A55203_cal_df),
    "A55204": calculate_chain_zero_degree_offsets(A55204_cal_df),
    "A55205": calculate_chain_zero_degree_offsets(A55205_cal_df),
}

# read high-precision thermometer data
hp_thermometer_dir = cal_dir + "/high_precision_thermometer.csv" # set path to high-precision thermometer data
hp_thermometer_df = pd.read_csv(hp_thermometer_dir, sep=';', header=0, decimal=',')

# apply correction from high-precision thermometer --> gives most accurate temperature of alcohol bath from which the offsets can be calculated
corrected_offsets = {}
for chain, offsets in all_offsets.items():
    bath_temp = hp_thermometer_df.loc[hp_thermometer_df['logger'] == chain, 'average temp'].values[0]
    corrected_offsets[chain] = {k: v - bath_temp for k, v in offsets.items()}

# generate variables holding corrected offsets per chain
A551FE_offsets = corrected_offsets['A551FE'] # AH1
A55204_offsets = corrected_offsets['A55204'] # AH2
A55205_offsets = corrected_offsets['A55205'] # AH3
A551FD_offsets = corrected_offsets['A551FD'] # HL1
A55203_offsets = corrected_offsets['A55203'] # HL2
A55200_offsets = corrected_offsets['A55200'] # HL3
A55201_offsets = corrected_offsets['A55201'] # CH1
A55202_offsets = corrected_offsets['A55202'] # CH2

# Convert the corrected_offsets dictionary to a DataFrame
offsets_df = pd.DataFrame.from_dict(corrected_offsets, orient='index')
offsets_df.index.name = 'chain'
offsets_df.reset_index(inplace=True)

# Save to CSV
offset_path = os.path.join(project_root, "data", "calibration", "corrected_chain_offsets.csv")
offsets_df.to_csv(offset_path, index=False)

# Convert the corrected_offsets dictionary to a DataFrame
offsets_df = pd.DataFrame.from_dict(corrected_offsets, orient='index')
offsets_df.index.name = 'chain'


## Calculate 0-degree offsets
Measurements in 0 degree alcohol bath. The final offsets will be calculated from the bath temperature measured by a high-precision thermometer.

## Plot Thermistor Chain Data

---

## Part 2 — Tynitag NTC Logger Calibration

## Import necessary libraries & Modules

In [ ]:
cal_path  = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "calibration_data") + "/"
out_figs  = os.path.join(project_root, "products", "figures", "thermistor_calibration") + "/"
out_csv   = cal_path + 'all_logger_offsets.csv'

In [ ]:
# 2) Logger -> calibration CSVs (adjust paths if needed)
logger_files = {
    1:  cal_path + 'calibration_runs_logger#1_#12/#1_ice_bath_0deg_offset_second_trial.csv',
    2:  cal_path + 'calibration_runs_logger#1_#12/#2_ice_bath_0deg_offset_second_trial.csv',
    3:  cal_path + 'calibration_runs_logger#1_#12/#3_ice_bath_0deg_offset_second_trial.csv',
    4:  cal_path + 'calibration_runs_logger#1_#12/#4_ice_bath_0deg_offset_second_trial.csv',
    5:  cal_path + 'calibration_runs_logger#1_#12/#5_ice_bath_0deg_offset_second_trial.csv',
    6:  cal_path + 'calibration_runs_logger#1_#12/#6_ice_bath_0deg_offset_second_trial.csv',
    7:  cal_path + 'calibration_runs_logger#1_#12/#7_ice_bath_0deg_offset_second_trial.csv',
    8:  cal_path + 'calibration_runs_logger#1_#12/#8_ice_bath_0deg_offset_second_trial.csv',
    9:  cal_path + 'calibration_runs_logger#1_#12/#9_ice_bath_0deg_offset.csv',
    10: cal_path + 'calibration_runs_logger#1_#12/#10_ice_bath_0deg_offset.csv',
    11: cal_path + 'calibration_runs_logger#1_#12/#11_ice_bath_0deg_offset.csv',
    12: cal_path + 'calibration_runs_logger#1_#12/#12_ice_bath_0deg_offset.csv',
    # # 13–16 reliability/extra runs
    13: cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/13_ice_bath_rel_exp3.csv',
    14: cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/14_ice_bath_rel_exp3.csv',
    15: cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/15_ice_bath_rel_exp3.csv',
    16: cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/16_ice_bath_rel_exp3.csv',
}

In [ ]:
# 3) Map each logger to the correct reference dataset
# These objects come from calibration.thermistor_calibration import *
# Adjust mapping if your design changed.
ref_by_logger = {
    1:  chains.data_10m_chain_2nd_ice_bath,
    2:  chains.data_10m_chain_2nd_ice_bath,
    3:  chains.data_10m_chain_2nd_ice_bath,
    4:  chains.data_10m_chain_2nd_ice_bath,
    5:  chains.data_10m_chain_2nd_ice_bath,
    6:  chains.data_10m_chain_2nd_ice_bath,
    7:  chains.data_10m_chain_2nd_ice_bath,
    8:  chains.data_10m_chain_2nd_ice_bath,
    9:  chains.data_10m_chain_3rd_ice_bath,
    10: chains.data_10m_chain_3rd_ice_bath,
    11: chains.data_10m_chain_4th_ice_bath,
    12: chains.data_10m_chain_4th_ice_bath,
    # 13: getattr(chains, "data_10m_chain_reliability_exp3", chains.data_10m_chain_4th_ice_bath),
    13: chains.data_10m_chain_4th_ice_bath,
    14: chains.data_10m_chain_4th_ice_bath,
    15: chains.data_10m_chain_4th_ice_bath,
    16: chains.data_10m_chain_4th_ice_bath,
}

In [ ]:
# 4) Run batch calibration and save outputs
df = compute_and_save_offsets(
    logger_files=logger_files,
    reference_by_logger=ref_by_logger,
    out_csv=out_csv,
    plots_dir=out_figs
)
df

In [ ]:
# 5) Quick QA
ax = df.plot(x="Logger", y=["Black Probe Offset", "White Probe Offset"], marker="o", grid=True)
ax.set_ylabel("Offset [°C]")
ax.figure.tight_layout()

## 6. Plot Reliability Experiments (Exp1, Exp2, Exp3)

Set file paths for reliability experiment data, create ThermistorDataPlotter objects, and plot calibration results for each experiment.

In [ ]:
# Experiment 1
logger_13_dir_exp1 = cal_path + 'NTC_reliability_experiments/Experiment1_20250130/NTCs/13_ice_bath_rel_exp1.csv'
logger_14_dir_exp1 = cal_path + 'NTC_reliability_experiments/Experiment1_20250130/NTCs/14_ice_bath_rel_exp1.csv'
logger_15_dir_exp1 = cal_path + 'NTC_reliability_experiments/Experiment1_20250130/NTCs/15_ice_bath_rel_exp1.csv'
logger_16_dir_exp1 = cal_path + 'NTC_reliability_experiments/Experiment1_20250130/NTCs/16_ice_bath_rel_exp1.csv'

logger_13_exp1 = ThermistorDataPlotter(logger_13_dir_exp1, delimiter=',')
logger_14_exp1 = ThermistorDataPlotter(logger_14_dir_exp1, delimiter=',')
logger_15_exp1 = ThermistorDataPlotter(logger_15_dir_exp1, delimiter=',')
logger_16_exp1 = ThermistorDataPlotter(logger_16_dir_exp1, delimiter=',')

zero_deg_offsets_logger13_exp1 = logger_13_exp1.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp1, savepath=out_figs, title='Logger #13 - 0deg offset in ice bath - Exp1')
zero_deg_offsets_logger14_exp1 = logger_14_exp1.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp1, savepath=out_figs, title='Logger #14 - 0deg offset in ice bath - Exp1')
zero_deg_offsets_logger15_exp1 = logger_15_exp1.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp1, savepath=out_figs, title='Logger #15 - 0deg offset in ice bath - Exp1')
zero_deg_offsets_logger16_exp1 = logger_16_exp1.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp1, savepath=out_figs, title='Logger #16 - 0deg offset in ice bath - Exp1')

# Experiment 2
logger_13_dir_exp2 = cal_path + 'NTC_reliability_experiments/Experiment2_20250304/NTCs/13_ice_bath_rel_exp2.csv'
logger_14_dir_exp2 = cal_path + 'NTC_reliability_experiments/Experiment2_20250304/NTCs/14_ice_bath_rel_exp2.csv'
logger_15_dir_exp2 = cal_path + 'NTC_reliability_experiments/Experiment2_20250304/NTCs/15_ice_bath_rel_exp2.csv'
logger_16_dir_exp2 = cal_path + 'NTC_reliability_experiments/Experiment2_20250304/NTCs/16_ice_bath_rel_exp2.csv'

logger_13_exp2 = ThermistorDataPlotter(logger_13_dir_exp2, delimiter=',')
logger_14_exp2 = ThermistorDataPlotter(logger_14_dir_exp2, delimiter=',')
logger_15_exp2 = ThermistorDataPlotter(logger_15_dir_exp2, delimiter=',')
logger_16_exp2 = ThermistorDataPlotter(logger_16_dir_exp2, delimiter=',')

zero_deg_offsets_logger13_exp2 = logger_13_exp2.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp2, savepath=out_figs, title='Logger #13 - 0deg offset in ice bath - Exp2')
zero_deg_offsets_logger14_exp2 = logger_14_exp2.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp2, savepath=out_figs, title='Logger #14 - 0deg offset in ice bath - Exp2')
zero_deg_offsets_logger15_exp2 = logger_15_exp2.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp2, savepath=out_figs, title='Logger #15 - 0deg offset in ice bath - Exp2')
zero_deg_offsets_logger16_exp2 = logger_16_exp2.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp2, savepath=out_figs, title='Logger #16 - 0deg offset in ice bath - Exp2')

# Experiment 3
logger_13_dir_exp3 = cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/13_ice_bath_rel_exp3.csv'
logger_14_dir_exp3 = cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/14_ice_bath_rel_exp3.csv'
logger_15_dir_exp3 = cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/15_ice_bath_rel_exp3.csv'
logger_16_dir_exp3 = cal_path + 'NTC_reliability_experiments/Experiment3_20250603/NTCs/16_ice_bath_rel_exp3.csv'

logger_13_exp3 = ThermistorDataPlotter(logger_13_dir_exp3, delimiter=',')
logger_14_exp3 = ThermistorDataPlotter(logger_14_dir_exp3, delimiter=',')
logger_15_exp3 = ThermistorDataPlotter(logger_15_dir_exp3, delimiter=',')
logger_16_exp3 = ThermistorDataPlotter(logger_16_dir_exp3, delimiter=',')

zero_deg_offsets_logger13_exp3 = logger_13_exp3.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp3, savepath=out_figs, title='Logger #13 - 0deg offset in ice bath - Exp3')
zero_deg_offsets_logger14_exp3 = logger_14_exp3.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp3, savepath=out_figs, title='Logger #14 - 0deg offset in ice bath - Exp3')
zero_deg_offsets_logger15_exp3 = logger_15_exp3.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp3, savepath=out_figs, title='Logger #15 - 0deg offset in ice bath - Exp3')
zero_deg_offsets_logger16_exp3 = logger_16_exp3.plot_ntc_icebath_calibration(chains.data_10m_chain_reliability_exp3, savepath=out_figs, title='Logger #16 - 0deg offset in ice bath - Exp3')

### Produce calibration statistical figures

In [ ]:
# Load offsets CSV
offsets_path = cal_path + 'all_logger_offsets.csv'
df_offsets = pd.read_csv(offsets_path)

# Only show loggers 1–15
df_offsets = df_offsets[df_offsets['Logger'] <= 15]

# Logger to borehole mapping
logger_to_borehole = {
    1: "SR1TT", 2: "SR2TT", 3: "GT1TT", 4: "GT2TT", 5: "HS1TT",
    6: "HS2TT", 7: "CJ1TT", 8: "CJ2TT", 9: "AH1TT", 10: "AH2TT",
    11: "CV1TT", 12: "CV2TT", 13: "AH3TT", 14: "CJ3TT", 15: "CJ4TT"
}

# integer ticks at each logger
ticks = sorted(df_offsets['Logger'].unique())

# Plot spread of offsets
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(df_offsets['Logger'], df_offsets['Black Probe Offset'], label='Deep probe', color='black', marker='o', s=100)
ax.scatter(df_offsets['Logger'], df_offsets['White Probe Offset'], label='Shallow probe', color='gray', marker='s', s=100)
ax.set_xlabel('Logger')
ax.set_ylabel('0°C Offset [°C]')
ax.grid(True, linestyle=':')
ax.legend(fontsize=14, edgecolor='black', frameon=True, fancybox=False)

ax.set_xticks(ticks)
ax.set_yticks(range(0, 7))  # from -1 to 1 degree
ax.set_xticklabels([int(t) for t in ticks])  # no decimals
ax.set_xlim(min(ticks) - 0.5, max(ticks) + 0.5)

# Plot median line for all offsets
all_offsets = pd.concat([df_offsets['Black Probe Offset'], df_offsets['White Probe Offset']])
median_offset = round(all_offsets.median(), 2)
ax.axhline(median_offset, color='red', linestyle='--', linewidth=2, label=f'Median 0°C offset ({median_offset:.2f}°C)')

# Move legend to avoid overlap with median line
ax.legend(fontsize=14, edgecolor='black', frameon=True, fancybox=False, loc='upper right')

# Create a legend dictionary for logger:borehole
legend_dict = {k: logger_to_borehole[k] for k in ticks}
legend_title = "Logger ↔ Borehole"
legend_text = f"{legend_title}\n" + "-"*len(legend_title) + "\n" + "\n".join([f"{k}: {v}" for k, v in legend_dict.items()])

# Place the legend on the right side of the plot with sharp, black-edged box
props = dict(boxstyle='square', facecolor='white', alpha=0.9, edgecolor='black', linewidth=1.5)
ax.text(1.02, 0.5, legend_text, transform=ax.transAxes, fontsize=14,
        verticalalignment='center', bbox=props, family='Arial')

plt.tight_layout()
plt.savefig(out_figs + 'ice_bath_offsets_spread_loggers_1-15.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Flatten all offsets from both probes into a single array, ignoring NaNs
all_offsets = pd.concat([df_offsets['Black Probe Offset'], df_offsets['White Probe Offset']]).dropna().values

plt.figure(figsize=(7, 4))
plt.boxplot(all_offsets, vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightgray', color='black'),
            medianprops=dict(color='red', linewidth=2))
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Offset [°C]')
plt.title('Spread of Tynitag Logger Offsets Around 0°C')
plt.tight_layout()
plt.show()

In [ ]:
# File paths
tynitag_dir = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "calibration_data", "all_logger_offsets.csv")
chain_dir = os.path.join(project_root, "data", "calibration", "corrected_chain_offsets.csv")

# Load offsets
df_tynitag = pd.read_csv(tynitag_dir)
df_chain = pd.read_csv(chain_dir, index_col='chain')

# Flatten offsets, ignoring NaNs and excluding values above 5
tynitag_offsets = pd.concat([df_tynitag['Black Probe Offset'], df_tynitag['White Probe Offset']]).dropna()
tynitag_offsets = tynitag_offsets[tynitag_offsets <= 5].values

chain_offsets = df_chain.values.flatten()
chain_offsets = chain_offsets[~pd.isnull(chain_offsets)]
chain_offsets = chain_offsets[chain_offsets <= 5]

# Calculate medians
median_tynitag = round(pd.Series(tynitag_offsets).median(), 3)
median_chain = round(pd.Series(chain_offsets).median(), 3)

# # no negative offset if value only 1/100 below zero
# if abs(median_chain) < 0.005:
#     median_chain = 0.0

# Set global font size and family
plt.rcParams.update({'font.size': 1, 'font.family': 'Arial'})

# Ensure global font size and family (override previous small size)
plt.rcParams.update({'font.size': 20, 'font.family': 'Arial'})

# Plot boxplots side by side
plt.figure(figsize=(8, 5))
box = plt.boxplot(
    [tynitag_offsets, chain_offsets],
    vert=False,
    patch_artist=True,
    widths=0.4,  # slightly wider boxes
    boxprops=dict(facecolor='lightgray', color='black', linewidth=1.5),
    medianprops=dict(color='red', linewidth=2),
    flierprops=dict(marker='o', markersize=7, markerfacecolor='white', markeredgecolor='black'),
    capprops=dict(linewidth=1.5),
    whiskerprops=dict(linewidth=1.5)
)
plt.yticks([1, 2], ['Tynitag', 'Geoprecision'], fontsize=20)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('0 °C Offset [°C]', fontsize=20)
plt.grid(True, linestyle=':')

# Add median lines and legend
plt.axvline(median_tynitag, color='k', linestyle='--', linewidth=2, label=f'TT median: {median_tynitag}°C')
plt.axvline(median_chain, color='k', linestyle=':', linewidth=2, label=f'G median: {median_chain}°C')
plt.legend(loc='upper right', frameon=True, edgecolor='black', fontsize=18, fancybox=False)

# Ensure tick label sizes
plt.tick_params(axis='both', which='major', labelsize=20)

plt.tight_layout()

plt.savefig(out_figs + 'tynitag_vs_geoprecision_offsets_boxplot.pdf', dpi=300, bbox_inches='tight')

out_figs_supp = os.path.join(project_root, 'figures', 'supplement', 'figS06_tynitag_vs_geoprecision_offsets_boxplot.pdf')
plt.savefig(out_figs_supp, dpi=300, bbox_inches='tight')

plt.show()